# 00 · Daten verstehen und den Test reservieren

**Ziel:** Ein gemeinsamer, überprüfter Ausgangspunkt. Noch keine Modellaufrufe, keine API-Kosten.

NLI beurteilt zwei Sätze: Folgt die zweite Aussage aus der ersten (**entailment**), widerspricht
sie ihr (**contradiction**), oder bleibt sie offen (**neutral**)? Neutral bedeutet nicht Unsicherheit.
ChaosNLI enthält bereits 100 menschliche Bewertungen pro Paar. Niemand muss neu annotieren.

Der Download verwendet einen dokumentierten, versionierten Spiegel von ChaosNLI sowie die
originalen Taxonomie-Dateien. Siehe `docs/DATA_PROVENANCE.md`. Beim ersten Lauf brauchst du Internet.
Installiere vorher die Pakete wie in der README beschrieben. Dann die Zellen von oben nach unten ausführen.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/project.json").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
from uncertainty_project.common import config
CFG = config(ROOT)
print("Projekt:", ROOT.name)

## Download und automatische Prüfungen

Die folgende Zelle prüft die Dateiprüfsummen, 100 Stimmen je Beispiel, Reihenfolge E/N/C,
fehlende Texte, doppelte IDs und den Anschluss der vorhandenen Kategorieannotationen.
Die Texte werden nicht sprachlich "bereinigt": Gerade Mehrdeutigkeit kann das Forschungssignal sein.

In [ ]:
from uncertainty_project.data import prepare_data, load_split, pilot_rows

audit = prepare_data(ROOT)
display(pd.Series({k:v for k,v in audit.items() if k != "taxonomy_leading_quote_differences"}))
print("Protokollierte Unterschiede bei führenden Anführungszeichen:",
      len(audit["taxonomy_leading_quote_differences"]))

## Drei getrennte Datenteile

**Training:** Hier darfst du Prompts ausprobieren und den kleinen Bewertungsbaustein trainieren.
Der Pilot mit 50 Beispielen kommt aus diesem Teil.

**Entwicklung (Dev):** Hier wählst du Baumgrösse und Reaktionsschwellen.

**Test:** Hier misst du erst am Schluss, wie gut die festgelegte Methode funktioniert.
Beispiele mit demselben Ausgangssatz bleiben im selben Teil. Die Aufteilung wird gespeichert
und beim erneuten Ausführen wiederverwendet. Testtexte und Testlabels liegen in getrennten Dateien.
Diese Trennung betrifft dein Experiment, nicht die unbekannten Vortrainingsdaten des Sprachmodells.

In [ ]:
train = load_split(ROOT, "train")
dev = load_split(ROOT, "dev")
test_inputs = load_split(ROOT, "test")  # Absichtlich ohne menschliche Labels
assert "human_distribution" not in test_inputs.columns
assert not (set(train.group_id) & set(dev.group_id))
assert not (set(train.group_id) & set(test_inputs.group_id))
assert not (set(dev.group_id) & set(test_inputs.group_id))
display(pd.DataFrame({"Teil":["Training", "Entwicklung", "Test"],
                      "Beispiele":[len(train),len(dev),len(test_inputs)]}))

## Ein echtes Trainingsbeispiel ansehen

Die folgende Verteilung besteht aus bereits erhobenen menschlichen Stimmen. Sie ist kein
GPT-Ergebnis. Viele verteilte Stimmen bedeuten Uneinigkeit; deren Ursache steht damit noch nicht fest.

In [ ]:
from uncertainty_project.common import LABELS
example = train.iloc[0]
print("Ausgangssatz:", example.premise)
print("Zu prüfende Aussage:", example.hypothesis)
display(pd.DataFrame({"Label":LABELS, "Stimmen":example.human_counts,
                      "Anteil":example.human_distribution}))
print("Vorhandene Kategorien:", example.taxonomy_categories)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from uncertainty_project.features import entropy

human = np.stack(train.human_distribution)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(entropy(human), bins=15, color="#1E2761", edgecolor="white")
ax.set(xlabel="Uneinigkeit (normierte Entropie)", ylabel="Trainingsbeispiele",
       title="Bereits erhobene menschliche Bewertungen")
plt.show()

## Was wir über einzelne Annotatoren nicht wissen

Wir sehen zum Beispiel die Anzahl der Stimmen für E, N und C. Wir sehen nicht, welche
derselben 100 Personen bei anderen Beispielen wie geantwortet hat. Deshalb lässt sich
aus diesen Dateien keine Liste zuverlässiger und unzuverlässiger Personen erstellen.
Die Taxonomie-Felder `a1` und `a2` sind Annotationen zu **Gründen** der Uneinigkeit,
nicht die fehlenden Identitäten der 100 ChaosNLI-Bewertenden.

**Nächster Schritt:** Notebook 01 mit dem 50er-Pilot. Halte den gewünschten Umgang mit
Annotatorenzuverlässigkeit als Scope-Frage für die Betreuung fest.

In [ ]:
pilot = pilot_rows(ROOT)
print(f"Pilot: {len(pilot)} Trainingsbeispiele, {len(pilot) * 5} geplante Modellantworten.")
print("Bisher wurden keine LLM-Vorhersagen erzeugt.")